In [6]:
import os
from typing import TypedDict, Annotated, Literal, List, Optional
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
import pandas as pd
llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:
class SimpleState(TypedDict):
    message : str

def greet(state: SimpleState) -> dict:
    return {'message' : f'안녕, {state["message"]}'}

builder = StateGraph(SimpleState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)

app = builder.compile()

In [3]:
app.invoke({'message':'abc'})

{'message': '안녕, abc'}

In [4]:
class TextState(TypedDict):
    text: str
    upper: str
    length: int

def to_upper(state: TextState) -> dict:
    return {'upper': state['text'].upper()}

def measure(state: TextState) -> dict:
    return {'length': len(state['text'])}

builder = StateGraph(TextState)
builder.add_node('to_upper', to_upper)
builder.add_node('measure', measure)
builder.add_edge(START, 'to_upper')
builder.add_edge('to_upper', 'measure')
builder.add_edge('measure', END)

app = builder.compile()

In [5]:
app.invoke({'text':'hello langgraph', 'upper': '', 'length':0})

{'text': 'hello langgraph', 'upper': 'HELLO LANGGRAPH', 'length': 15}

In [7]:
class QueryAnalysis(BaseModel):
    query : str = Field(description='원본쿼리')
    route : Literal['direct', 'rag', 'web_search'] = Field(
        description='라우팅 경로: direct(직접 답변), rag(문서 검색), web_search(웹 검색)'
    )
    confidence : float = Field(description='분류 신뢰도(0.0~0.1)', ge=0.0, le=1.0)
    reasoning : str = Field(description='분류 이유')

In [10]:
def analyze_query(query:str) -> QueryAnalysis:
    """LLM을 사용하여 쿼리를 분석하고 라우팅 경로를 결정"""
    system_prompt = """당신은 쿼리를 잘 분석하고 라우팅 경로를 분류하는 전문가입니다. 주어진 쿼리를 분석해서 최적의 처리 경로를 분석하세요
    
    라우팅 경로: 
    direct : LLM 이 자체 지식으로 답변 가능한 일반 질문(상식, 개념 설명)
    rag : 회사 내부 문서나 특정 도메인 지식이 필요한 질문(정책, 메뉴얼, 사내 데이터)
    web_search : 최신 정보나 실시간 데이터가 필요한 질문(뉴스, 시세, 날씨)

    예시:
    - 파이썬 데코레이터란? -> direct (일반 프로그래밍 지식)
    - 우리 회사 연차 규정은? -> rag (내부 문서 필요)
    - 오늘 날씨는? -> web_search (실시간 정보 필요)
    """

    response = llm.with_structured_output(QueryAnalysis).invoke([SystemMessage(content=system_prompt), HumanMessage(content=query)])

    return response

In [11]:
analyze_query("1+1은?")

QueryAnalysis(query='1+1은?', route='direct', confidence=0.9, reasoning='이 질문은 기본적인 산수 문제로, LLM의 자체 지식으로 쉽게 답변할 수 있습니다.')

In [12]:
from dataclasses import dataclass, field
from datetime import datetime

In [17]:
@dataclass
class RoutingConfig:
    """라우팅 전략을 설정"""
    confidence_threshold : float = 0.7
    fallback_route : str = 'rag'
    enable_logging : bool = True
    max_retires : int = 2

@dataclass
class RoutingLog:
    """라우팅 로그"""
    timestamp : str
    query : str
    prediceted_route : str
    actual_route : str
    confidence : float
    fallback_applied : bool

In [20]:
class RoutingEngine:
    def __init__(self, config: RoutingConfig = None):
        self.config = config or RoutingConfig()
        self.logs : list[RoutingLog] = []

    def route(self, analysis: QueryAnalysis) -> str:
        """분석 결과를 바탕으로 최종 라우팅 경로를 결정"""
        predicted = analysis.route
        fallback_applied = False

        if analysis.confidence < self.config.confidence_threshold:
            actual = self.config.fallback_route
            fallback_applied = True
        else:
            actual = predicted

        if self.config.enable_logging:
            log = RoutingLog(timestamp=datetime.now().isoformat(), query=analysis.query,
                            prediceted_route=predicted,
                             actual_route=actual,
                             confidence=analysis.confidence,
                             fallback_applied = fallback_applied
                            )
            self.logs.append(log)
        return actual

    def get_stats(self) -> dict:
        if not self.logs:
            return {'total' : 0}
        total = len(self.logs)
        fallbacks = sum(1 for log in self.logs if log.fallback_applied)
        route_dist = {}
        for log in self.logs:
            route_dist[log.actual_route] = route_dist.get(log.actual_route, 0) + 1
        avg_conf = sum(log.confidence for log in self.logs) / total

        return {
            'total': total,
            'fallback_rate' : fallback/total,
            'avg_confidence' : avg_conf,
            'route_distribution' : route_dist
        }

In [21]:
engine = RoutingEngine(RoutingConfig(confidence_threshold=0.7))

In [22]:
analysis = analyze_query('파이썬 데코레이터가 뭔가요?')
final_route = engine.route(analysis)

In [24]:
from abc import ABC, abstractmethod

In [25]:
class RouteHandler(ABC):
    @abstractmethod
    def handle(self, query: str) -> str:
        pass

In [26]:
class DirectHandler(RouteHandler):
    """LLM이 직접 답변하는 경우"""
    def handle(self, query:str) -> str:
        response = llm.invoke([HumanMessage(content=query)])
        return response.content

class RAGHandler(RouteHandler):
    """RAG 검색해서 답변하는 경우"""
    def handle(self, query:str) -> str:
        return f"RAG {query}에 대한 문서 검색 결과 입니다."


class WebSearchHandler(RouteHandler):
    """웹 검색해서 답변하는 경우"""
    def handle(self, query:str) -> str:
        return f"WebSearch {query}에 대한 문서 검색 결과 입니다."

In [27]:
handlers = {
    'direct' : DirectHandler(),
    'rag' : RAGHandler(),
    'web_search' : WebSearchHandler()
}

In [28]:
test_q = '파이썬에서 리스트 컴프리헨션이란?'
analysis = analyze_query(test_q)
route = engine.route(analysis)
result = handlers[route].handle(test_q)
print(result)

파이썬에서 리스트 컴프리헨션(list comprehension)은 기존의 리스트를 기반으로 새로운 리스트를 간결하게 생성하는 방법입니다. 전통적인 방법에 비해 더 간결하고 읽기 쉬운 코드로 리스트를 만들 수 있는 장점이 있습니다.

리스트 컴프리헨션의 기본 구문은 다음과 같습니다:

```python
[표현식 for 항목 in iterable if 조건]
```

- `표현식`: 새로운 리스트의 각 요소를 생성하기 위한 표현식입니다.
- `항목`: iterable에서 가져온 각각의 요소입니다.
- `iterable`: 리스트, 튜플, 문자열 등 반복 가능한 객체입니다.
- `조건` (선택적): 특정 조건을 만족하는 경우에만 리스트에 포함시키고자 할 때 사용합니다.

### 예제

1. **기본적인 리스트 컴프리헨션**:
   ```python
   squares = [x**2 for x in range(10)]
   print(squares)  # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
   ```

2. **조건문을 포함한 리스트 컴프리헨션**:
   ```python
   even_squares = [x**2 for x in range(10) if x % 2 == 0]
   print(even_squares)  # [0, 4, 16, 36, 64]
   ```

3. **문자열에서 리스트 만들기**:
   ```python
   input_string = "hello"
   char_list = [char.upper() for char in input_string]
   print(char_list)  # ['H', 'E', 'L', 'L', 'O']
   ```

리스트 컴프리헨션은 파이썬의 강력한 기능 중 하나로, 코드의 가독성을 높이고, 짧은 코드로 효율적인 작업을 수행할 수 있게 해줍니다.


In [30]:
@dataclass
class AdvancedRoutingConfig:
    confidence_threshold: float = 0.5
    fallback_route: str = "rag"
    route_weights: dict = field(default_factory=lambda: {
        "direct": 1.0, "rag": 0.8, "web_search": 0.6
    })

In [31]:
class WeightedRoutingEngine:
    def init(self, config: AdvancedRoutingConfig = None):
        self.config = config or AdvancedRoutingConfig()
    def weighted_route(self, analysis: QueryAnalysis) -> dict:
        weight = self.config.route_weights.get(analysis.route, 1.0)
        weighted_score = analysis.confidence * weight
        if weighted_score < self.config.confidence_threshold:
            final_route = self.config.fallback_route
            fallback = True
        else:
            final_route = analysis.route
            fallback = False
        return {
            "query": analysis.query[:25],
            "predicted": analysis.route,
            "confidence": analysis.confidence,
            "weight": weight,
            "weighted_score": round(weighted_score, 3),
            "final_route": final_route,
            "fallback": fallback
        }